# **MamaCare-AI Maternal Triage**


### **Install Environment Dependencies**

In [1]:
!pip install -q \
    transformers==4.44.2 \
    peft==0.12.0 \
    setfit==1.1.0 \
    datasets==2.20.0 \
    sentence-transformers==3.0.1 \
    --force-reinstall)

/bin/bash: -c: line 1: syntax error near unexpected token `)'
/bin/bash: -c: line 1: `pip install -q      transformers==4.44.2      peft==0.12.0      setfit==1.1.0      datasets==2.20.0      sentence-transformers==3.0.1      --force-reinstall)'


### **Data Laoding**

In [1]:
import json
import pandas as pd
from datasets import Dataset


print("Loading custom triage anchors...")
with open("/kaggle/input/datasets/sammydamz/t-system/triage_anchors.json", "r", encoding="utf-8") as f:
    anchors_data = json.load(f)


df = pd.DataFrame(anchors_data)
df = df[['text', 'label']]


df_oversampled = pd.concat([df] * 10, ignore_index=True)

print(f"\nOriginal Dataset Size: {len(df)} rows")
print(f"Oversampled Dataset Size: {len(df_oversampled)} rows")
print(df_oversampled['label'].value_counts())

Loading custom triage anchors...

Original Dataset Size: 28 rows
Oversampled Dataset Size: 280 rows
label
0    130
2    120
1     30
Name: count, dtype: int64


### **Step 3: Tokenization & Model Loading**
to load multilingual checkpoint, tokenizes the dataset, and loads the sequence classification model with 3 output labels (LOW, MEDIUM, HIGH).

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint = "distilbert-base-multilingual-cased"
print(f"Loading tokenizer and model for {model_checkpoint}...")

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Convert pandas DataFrame to Hugging Face Dataset and tokenize
dataset = Dataset.from_pandas(df_oversampled)
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.shuffle(seed=42)

# Load sequence classification model (3 classes: 0=LOW, 1=MEDIUM, 2=HIGH)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

Loading tokenizer and model for distilbert-base-multilingual-cased...


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### **Step 4: Run Fine-Tuning**
Configures `TrainingArguments` and trains the model for 3 epochs using the standard `Trainer`.

In [4]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("Starting model training...")
trainer.train()
print("Training completed!")

Starting model training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,1.933076
20,1.554133
30,1.092241
40,0.766761
50,0.604286


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed!


### **Step 5: Test Model Predictions & Map Guidelines**
Evaluates the model and computes semantic similarity against your anchors to map the exact clinical guideline used to make the triage decision.

In [6]:
import torch

test_phrases = [
    "I am bleeding heavily since this morning",
    "I feel fine, just a bit tired",
    "I have a mild headache but my vision is normal",
    "My feet and hands are swollen and I cannot see clearly",
    "I am fine no issues"
]

model.eval()
labels_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

print("\n--- Test Predictions ---")
for phrase in test_phrases:
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    print(f"Text: '{phrase}' -> Predicted Risk: {labels_map[prediction]}")



--- Test Predictions ---
Text: 'I am bleeding heavily since this morning' -> Predicted Risk: HIGH
Text: 'I feel fine, just a bit tired' -> Predicted Risk: LOW
Text: 'I have a mild headache but my vision is normal' -> Predicted Risk: HIGH
Text: 'My feet and hands are swollen and I cannot see clearly' -> Predicted Risk: HIGH
Text: 'I am fine no issues' -> Predicted Risk: LOW


In [7]:
import json
import torch
from sklearn.metrics import classification_report, accuracy_score


print("Loading evaluation test data...")
with open("/kaggle/input/datasets/sammydamz/t-system/triage_test_data.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_texts = [item["text"] for item in test_data]
true_labels = [item["expected_label"] for item in test_data]
labels_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

# Run predictions
predicted_labels = []
model.eval()

print("Running evaluation inference...")
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    predicted_labels.append(pred)

# Calculate and display metrics
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"\nModel Accuracy on Test Set: {accuracy * 100:.2f}%")

print("\n--- Detailed Classification Report ---")
print(classification_report(true_labels, predicted_labels, target_names=["LOW", "MEDIUM", "HIGH"]))


Loading evaluation test data...
Running evaluation inference...

Model Accuracy on Test Set: 66.67%

--- Detailed Classification Report ---
              precision    recall  f1-score   support

         LOW       0.56      1.00      0.71         5
      MEDIUM       1.00      0.20      0.33         5
        HIGH       0.80      0.80      0.80         5

    accuracy                           0.67        15
   macro avg       0.79      0.67      0.62        15
weighted avg       0.79      0.67      0.62        15



### **Step 6: Push Model to Hugging Face Hub**
Prompts login to Hugging Face and pushes model weights and tokenizers to the Hub.

In [10]:
from huggingface_hub import notebook_login

notebook_login()

# Save tokenizer and model locally
model.save_pretrained("./mamacare-triage-model")
tokenizer.save_pretrained("./mamacare-triage-model")


model.push_to_hub("sammydamz/mamacare-triage-model")
tokenizer.push_to_hub("sammydamz/mamacare-triage-model")
print("Model successfully uploaded to Hugging Face Hub!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model successfully uploaded to Hugging Face Hub!
